In [ ]:
fn = "../saves/reproduce/gsap_models/pruner/biafencoder-spanlen12-rank768-hid768-span256-entnum3-18-lr2e-5-epochs8/scierc_scibert-44/test_spkogpu01.log",

In [ ]:
fn = "../saves/reproduce/gsap_models/pruner/biafencoder-spanlen12-rank768-hid768-span256-entnum3-20-lr2e-5-epochs8/scierc_scibert-44/test_results.txt

In [2]:
fn = "../saves/scinlp/*"
from glob import glob
glob(fn)


[]

In [192]:
log = open(fn).read()

In [193]:
print(log[-5000:])

 "of", "true", "three-dimensional", "affine", "and", "Euclidean", "models", "from", "multiple", "images", "and", "their", "recognition", "in", "a", "single", "photograph", "taken", "from", "an", "arbitrary", "viewpoint", "."], ["The", "proposed", "approach", "does", "not", "require", "a", "separate", "segmentation", "stage", "and", "is", "applicable", "to", "cluttered", "scenes", "."], ["Preliminary", "modeling", "and", "recognition", "results", "are", "presented", "."]], "ner": [[[5, 5, "Generic"], [7, 8, "OtherScientificTerm"], [12, 14, "OtherScientificTerm"], [17, 18, "OtherScientificTerm"]], [[20, 21, "OtherScientificTerm"], [31, 32, "Method"], [38, 38, "Task"], [40, 40, "Task"], [44, 51, "Task"], [54, 54, "Material"]], [[70, 70, "Generic"], [76, 77, "Method"], [82, 83, "OtherScientificTerm"]], [[88, 88, "Task"]]], "relations": [[[5, 5, 7, 8, "USED-FOR"], [12, 14, 7, 8, "FEATURE-OF"], [17, 18, 12, 14, "FEATURE-OF"]], [[20, 21, 31, 32, "CONJUNCTION"], [20, 21, 38, 38, "USED-FOR"], [

In [ ]:
fn_train = "../saves/reproduce/gsap_models/pruner/biafencoder-spanlen12-rank768-hid768-span256-entnum3-24-lr2e-5-epochs8/scierc_scibert-44/ent_pred_train.json"

In [200]:
fn_train = "../saves/reproduce/sciner_models/pruner/biafencoder-spanlen12-rank768-hid768-span256-entnum3-18-lr2e-5/scierc_scibert-43/ent_pred_test_overlap.json"

In [215]:
import json
import random
from itertools import chain

In [216]:
train_docs = []
with open(fn_train) as f:
    for line in f.readlines():
        doc = json.loads(line)
        if "doc_tokens" not in doc:
            doc["doc_tokens"] = list(chain(*doc["sentences"]))
        train_docs.append(doc)
len(train_docs)

100

In [217]:
doc.keys()

dict_keys(['clusters', 'sentences', 'ner', 'relations', 'doc_key', 'predicted_ner', 'doc_tokens'])

In [218]:
sent_begin = 0
for doc in train_docs:
    for sent_idx, sentence in enumerate(doc["sentences"]):
        sent_ner_preds = doc["predicted_ner"][sent_idx]
        for begin, end, label in sent_ner_preds:
            if begin >= sent_begin + len(sentence):
                print("ihh")
        sent_begin += len(sentence)

In [221]:
doc = random.choice(train_docs)
sent_idxs = list(range(len(doc["sentences"])))
sent_infos = []
for sent_idx in sent_idxs:
    sentence = doc["sentences"][sent_idx]
    ner_pred = doc["predicted_ner"][sent_idx]
    ner = doc["ner"][sent_idx]
    ner_set = {(begin, end) for begin, end, _ in ner}
    ner_pred_set = {(begin, end) for begin, end, _ in ner_pred}
    recall=None
    if ner_set:
        tp = len(ner_set & ner_pred_set)
        fn = len(ner_set - ner_pred_set)
        fp = len(ner_pred_set - ner_set)
        recall = tp / len(ner_set)
        #recall, tp, fn, fp
    sent_infos.append(dict(sent_idx=sent_idx, recall=recall))
len(sent_idxs)

7

In [222]:
import pandas as pd
pd.DataFrame(sent_infos).recall.value_counts().sort_index()

1.0    7
Name: recall, dtype: int64

In [223]:
sent_infos_fn = [i for i in sent_infos if i["recall"] is not None and i["recall"] <= 1.]
len(sent_infos_fn)

7

In [224]:
tp = len(ner_set & ner_pred_set)
fn = len(ner_set - ner_pred_set)
fp = len(ner_pred_set - ner_set)
recall = tp / len(ner_set)
recall, tp, fn, fp


(1.0, 1, 0, 8)

In [225]:
sent_info = random.choice(sent_infos_fn)
print(sent_info)
sent_idx = sent_info["sent_idx"]
sentence = doc["sentences"][sent_idx]
ner_pred = doc["predicted_ner"][sent_idx]
ner = doc["ner"][sent_idx]

print(" ".join(sentence))
print("\nner:")
for begin, end, label in ner:
    print(f'[{label}: "{" ".join(doc["doc_tokens"][begin:end+1])}"]')
print("\npredicted ner candidates:")
for begin, end, label in ner_pred:
    print(f'[{label}: "{" ".join(doc["doc_tokens"][begin:end+1])}"]')

{'sent_idx': 5, 'recall': 1.0}
To our knowledge , this is the first a posteriori bound for joint matrix decomposition .

ner:
[OtherScientificTerm: "posteriori bound"]
[Task: "joint matrix decomposition"]

predicted ner candidates:
[NIL: ", this"]
[NIL: "this"]
[NIL: "first a posteriori bound"]
[NIL: "a posteriori bound"]
[NIL: "a posteriori bound for joint matrix decomposition"]
[OtherScientificTerm: "posteriori bound"]
[Task: "joint matrix decomposition"]
[NIL: "joint matrix decomposition ."]
[NIL: "matrix decomposition"]


In [57]:
for begin, end, label in ner_pred:
    print(doc["doc_tokens"][begin:end+1], label)

['Instances'] NIL
['few', 'features'] NIL
['learning'] NIL
['accurate', 'feature', 'predictors'] NIL
['feature', 'predictors'] MLModelGeneric
